# 04 — Evaluation

**Model evaluation.** Loads the trained baseline from `outputs/models/` and evaluates on the held-out test split: precision/recall/F1, confusion matrix, and error analysis.

## Setup

In [ ]:
import sys, os

PROJECT_PATH = "/content/Sexism-Classification" if os.path.exists("/content/Sexism-Classification") else os.getcwd()
sys.path.append(PROJECT_PATH)

import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

In [ ]:
from src.config.settings import PROCESSED_DATA_DIR, TEXT_COLUMN, LABEL_COLUMN
from src.models.predict import _load_components
from src.data.preprocess import clean_text

model, vectorizer = _load_components()
print("Model and vectorizer loaded.")

## Evaluate on held-out data

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from src.config.settings import RANDOM_SEED
from src.models.utils import evaluate_model

df = pd.read_csv(PROCESSED_DATA_DIR / "train.csv")
df["clean"] = df[TEXT_COLUMN].apply(clean_text)

# Same stratified split (and seed) as training -> identical held-out set
_, test_df = train_test_split(df, test_size=0.2, random_state=RANDOM_SEED, stratify=df[LABEL_COLUMN])

X_test = vectorizer.transform(test_df["clean"])
y_test = test_df[LABEL_COLUMN]

metrics = evaluate_model(model, X_test, y_test)
print("Accuracy:", round(metrics["accuracy"], 4))
print("F1 (weighted):", round(metrics["f1"], 4))
print()
print(metrics["report"])

## Confusion matrix

In [ ]:
preds = model.predict(X_test)
cm = confusion_matrix(y_test, preds, labels=model.classes_)

plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=model.classes_, yticklabels=model.classes_)
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion matrix — TF-IDF + LogReg baseline")
plt.show()

## Error analysis

Inspect the misclassified examples to spot preprocessing gaps, label noise, or hard cases (e.g. reclaimed slurs, quoted speech, sarcasm).

In [ ]:
test_df = test_df.assign(pred=preds)
errors = test_df[test_df[LABEL_COLUMN] != test_df["pred"]]
print(f"{len(errors)} misclassified of {len(test_df)}")
errors[[TEXT_COLUMN, LABEL_COLUMN, 'pred']].head(20)